<a href="https://colab.research.google.com/github/martinruhle/curso-ciencia-datos-2027-1/blob/lab02/lab02/analisis_pacientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 02 — Análisis de un dataset de pacientes con presupuesto de memoria

**Actividades 1–3.** Dataset sintético generado con Synthea (**23 018 pacientes**: 20 001 vivos + 3 017 fallecidos).

In [1]:
FORCE_REBUILD = False   # True = ignorar caché y correr el pipeline completo

CKPT = BASE / 'ckpt'
usar_cache = (not FORCE_REBUILD) and (CKPT/'obs.parquet').exists()

if usar_cache:
    obs     = pd.read_parquet(CKPT/'obs.parquet')
    p8      = pd.read_parquet(CKPT/'p8.parquet')
    enc_tip = pd.read_parquet(CKPT/'enc_tip.parquet')
    print('estado restaurado desde checkpoint')
    print(obs.dtypes)        # verificá que category/float32 sobrevivieron
else:
    print('sin caché: corré las celdas de Actividades 0-3')

NameError: name 'BASE' is not defined

## 0. Entorno y datos

### 0.1 Librerías
`polars`, `pyspark`, `memory_profiler` y `pyarrow` no vienen en Colab; el resto sí. `gc` se usa para liberar RAM a mano entre la carga y la selección de datatype de `observations`, que es la tabla pesada (17 millones de filas).

In [1]:
!pip install -q polars pyspark memory_profiler pyarrow
import pandas as pd, numpy as np, gc

### 0.2 Persistencia en Drive
Colab borra el disco al reiniciar. Montamos Drive y fijamos ahí las rutas para no regenerar los datos en cada sesión.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/curso-cd-2027/lab02')
DATA = BASE / 'data'
DATA.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


### 0.3 Generación con Synthea
Se descarga el jar `with-dependencies` desde el tag `master-branch-latest` (siempre apunta a la última versión). Dos decisiones en el comando de generación:

- `--exporter.csv.export true` — enciende la exportación a CSV, apagada por defecto.
- `--exporter.fhir.export false` — apaga FHIR, que no se usa en este lab y triplica el tiempo.

`-p 20001` fija la población **viva**; Synthea además simula a quienes murieron en la ventana temporal, así que `patients.csv` termina con **23 018 filas** (20 001 vivas + 3 017 fallecidas). El comando de abajo es el exacto usado, anotado por reproducibilidad.

In [ ]:
# Java (Synthea necesita 11+)
!java -version 2>&1 | head -1 || apt-get -qq install -y openjdk-17-jdk-headless

# Jar "with dependencies" desde Releases:
!wget -q https://github.com/synthetichealth/synthea/releases/download/master-branch-latest/synthea-with-dependencies.jar -O synthea-with-dependencies.jar

In [ ]:
# Comando de generación (reproducibilidad):
!java -jar synthea-with-dependencies.jar -p 20001 --exporter.csv.export true --exporter.fhir.export false

### 0.4 Verificación de salida
Confirmamos que los tres CSV que usa el lab estén en `output/csv/` **antes** de seguir.

In [ ]:
!ls -la output/csv/

ls: cannot access '/data': No such file or directory


### 0.5 Copia a Drive

In [ ]:
!cp -r output/csv/*.csv "{DATA}/"
sorted(p.name for p in DATA.glob('*.csv'))

cp: cannot stat 'output/csv/*.csv': No such file or directory


['allergies.csv',
 'careplans.csv',
 'claims.csv',
 'claims_transactions.csv',
 'conditions.csv',
 'devices.csv',
 'encounters.csv',
 'imaging_studies.csv',
 'immunizations.csv',
 'medications.csv',
 'observations.csv',
 'organizations.csv',
 'patients.csv',
 'payer_transitions.csv',
 'payers.csv',
 'procedures.csv',
 'providers.csv',
 'supplies.csv']

## Actividad 1 — Carga con dtypes explícitos

Cargamos cada archivo dos veces —primero con la inferencia de pandas, después declarando los dtypes— y medimos el consumo en cada caso.

### Clasificación de columnas (`patients.csv`)
El dtype sale de clasificar cada columna en una de tres cubetas:

| Columna | Cubeta | dtype |
|---|---|---|
| `Id` | identificador (único, 1 por fila) | `object` |
| `BIRTHDATE` | fecha | `datetime64` |
| `DEATHDATE` | fecha | `datetime64` |
| `GENDER` | categórica | `category` |
| `RACE` | categórica | `category` |
| `ETHNICITY` | categórica | `category` |
| `CITY` | categórica (mayor cardinalidad) | `category` |
| `STATE` | categórica (cardinalidad 1: todo Massachusetts) | `category` |

> Sobre `CODE` (en encounters/observations): *parece* identificador, pero es un código clínico (SNOMED/LOINC) con pocas decenas de valores que se repiten en millones de filas → `category`, no `object`.

### Medición de memoria
`memory_usage(deep=True)` es la medición correcta: sin `deep`, una columna `object` reporta solo el peso de los punteros (8 bytes por celda), no los bytes reales de cada string. En columnas `object` la diferencia es de órdenes de magnitud.

In [3]:
def mb(df):
    """Memoria real del DataFrame en MB."""
    return df.memory_usage(deep=True).sum() / 1e6

### Carga ingenua (baseline)
Primera pasada: dtypes inferidos, anotando el consumo de los tres archivos. El `del ... gc.collect()` libera antes de seguir — clave con `observations`.

In [4]:
naive = {}
for name in ['patients', 'encounters', 'observations']:
    df = pd.read_csv(DATA / f'{name}.csv')     # inferencia por defecto
    naive[name] = mb(df)
    print(f'{name:14s} {naive[name]:8.1f} MB   {df.shape}')
    del df; gc.collect()

patients           28.3 MB   (23018, 28)
encounters       1099.1 MB   (1355775, 15)
observations    10454.3 MB   (17355578, 9)


### Tabla comparativa: crudo (28 col) vs filtrado (8 col) vs tipado
`usecols` filtra **en la lectura**, no después. Comparamos las 28 columnas que exporta Synthea contra las 8 que pide el lab, y esas 8 contra su versión tipada. `enc_naive` queda cargado para reutilizarlo en la Actividad 3.

In [5]:
mem = []  # filas de la tabla comparativa

# ---- patients ----
cols_patients = ['Id','BIRTHDATE','DEATHDATE','GENDER','RACE','ETHNICITY','CITY','STATE']
dtypes_patients = {'Id':'object','GENDER':'category','RACE':'category',
                   'ETHNICITY':'category','CITY':'category','STATE':'category'}
# BIRTHDATE/DEATHDATE van en parse_dates, NO en dtype (no pueden estar en ambos)

p_raw   = pd.read_csv(DATA/'patients.csv')                       # 28 col, inferido
mem.append(('patients','crudo 28col', mb(p_raw), p_raw.shape));  del p_raw; gc.collect()

p8_naive = pd.read_csv(DATA/'patients.csv', usecols=cols_patients)   # 8 col, inferido
mem.append(('patients','8col inferido', mb(p8_naive), p8_naive.shape))

p8 = pd.read_csv(DATA/'patients.csv', usecols=cols_patients,
                 dtype=dtypes_patients, parse_dates=['BIRTHDATE','DEATHDATE'])  # 8 col tipado
mem.append(('patients','8col tipado', mb(p8), p8.shape))

# ---- encounters ----  mismo criterio: ID=object, categóricas, fechas
cols_enc = ['Id','START','STOP','PATIENT','ENCOUNTERCLASS','CODE','DESCRIPTION']
enc_naive = pd.read_csv(DATA/'encounters.csv', usecols=cols_enc)
mem.append(('encounters','inferido', mb(enc_naive), enc_naive.shape))

dtypes_enc = {'Id':'object','PATIENT':'category','ENCOUNTERCLASS':'category',
              'CODE':'category','DESCRIPTION':'category'}   # PATIENT repite: category
enc_tip = pd.read_csv(DATA/'encounters.csv', usecols=cols_enc,
                      dtype=dtypes_enc, parse_dates=['START','STOP'])
mem.append(('encounters','tipado', mb(enc_tip), enc_tip.shape))

# ---- observations ----  solo baseline; la reducción es la Act.2
obs_naive = pd.read_csv(DATA/'observations.csv')
mem.append(('observations','inferido', mb(obs_naive), obs_naive.shape))

tabla1 = pd.DataFrame(mem, columns=['archivo','version','MB','shape'])
tabla1

,archivo,version,MB,shape
0,patients,crudo 28col,28.292863,"(23018, 28)"
1,patients,8col inferido,10.645683,"(23018, 8)"
2,patients,8col tipado,2.505041,"(23018, 8)"
3,encounters,inferido,622.489679,"(1355775, 7)"
4,encounters,tipado,146.211358,"(1355775, 7)"
5,observations,inferido,10454.268422,"(17355578, 9)"


Si bien el filtrado de las columnas que no pertenecen a las importantes es un paso fundamental (ya que se ahorra casi 2 tercios de la memoria), podemos observar que el mayor ahorro viene dado por la declaración de los data types (en donde se ahorra el 80%, aprox, de la memoria para los archivos `patientes` y `encounters`). Para el archivo obervations vamos a realizar el mismo análisis en la actividad siguiente.

## Actividad 2 — Reducir la memoria de `observations` ≥ 70%

### Diagnóstico antes de decidir
Antes de convertir nada: cardinalidad por columna (decide dónde `category` ayuda) y qué fracción de `VALUE` no es numérica.

In [6]:
base_mb = tabla1.query("archivo=='observations' and version=='inferido'")['MB'].iloc[0]

# 1) cardinalidad por columna
n = len(obs_naive)
for c in obs_naive.columns:
    print(f'{c:12s} nunique={obs_naive[c].nunique():>8}  ({obs_naive[c].nunique()/n:.1%} de {n})')

# 2) VALUE: fracción NO numérica y qué hay ahí
val_num = pd.to_numeric(obs_naive['VALUE'], errors='coerce')
print('\nVALUE no-numérico:', f'{val_num.isna().mean():.1%}')
print(obs_naive.loc[val_num.isna(),'VALUE'].value_counts().head(10))
print('\nTYPE:'); print(obs_naive['TYPE'].value_counts())

DATE         nunique= 1793984  (10.3% de 17355578)
PATIENT      nunique=   23018  (0.1% de 17355578)
ENCOUNTER    nunique=  732369  (4.2% de 17355578)
CATEGORY     nunique=       8  (0.0% de 17355578)
CODE         nunique=     298  (0.0% de 17355578)
DESCRIPTION  nunique=     300  (0.0% de 17355578)
VALUE        nunique=   45621  (0.3% de 17355578)
UNITS        nunique=      51  (0.0% de 17355578)
TYPE         nunique=       2  (0.0% de 17355578)

VALUE no-numérico: 36.9%
VALUE
No                                         1610881
Yes                                         401400
I have housing                              225943
Never smoked tobacco (finding)              222206
English                                     206371
White                                       185421
I choose not to answer this question        165953
Full-time work                              148081
Cloudy urine (finding)                      147319
Finding of bilirubin in urine (finding)     147015
Name: c

### Por qué `VALUE` está mezclada
`VALUE` viene como texto porque combina resultados numéricos con categóricos. Se ve mirando filas y un caso claro (`Tobacco smoking status`), donde `VALUE` es una categoría nominal y `UNITS` es `NaN`.

In [ ]:
print(obs_naive.head(25))

                    DATE                               PATIENT  \
0   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
1   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
2   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
3   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
4   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
5   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
6   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
7   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
8   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
9   2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
10  2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
11  2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
12  2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
13  2018-08-17T21:58:51Z  e82ce5d7-a72c-7886-8c2b-42b8641f8448   
14  2018-0

In [7]:
valores = obs_naive.loc[obs_naive["DESCRIPTION"] == "Tobacco smoking status", "VALUE"].unique()
print(valores)

['Never smoked tobacco (finding)' 'Ex-smoker (finding)'
 'Smokes tobacco daily (finding)']


### Decisión de diseño para `VALUE`
Se evaluaron dos opciones:

1. Codificar todos los textos como enteros con leyenda en `UNITS` → **descartada**: la mayoría de los textos son nominales sin orden (`English`, `White`, `Yes/No`); la codificación tendría que ser por `DESCRIPTION` (298 codebooks); y `UNITS` no es un almacén de metadata.
2. **Partir `VALUE` en `VALUE_num` (float32) + `VALUE_cat` (category) y descartar el original.** ← elegida

La partición usa `TYPE` (ground truth de Synthea: `numeric`/`text`) para decidir fila por fila. Coincide con cómo OMOP CDM modela esto: `value_as_number` / `value_as_concept_id` en columnas separadas.

De esta manera ahorramos con las celdas de `VALUE` que eran numéricas con `float32` (alrededor de un 60% de las celdas) y con el resto con `category`.

### Implementación

In [8]:
def col_mb(df):
    return (df.memory_usage(deep=True)/1e6).round(3)  # MB por columna

antes = col_mb(obs_naive)

obs = obs_naive.copy()
obs['DATE'] = pd.to_datetime(obs['DATE'], utc=True)

for c in ['CATEGORY','CODE','DESCRIPTION','UNITS','TYPE']:
    obs[c] = obs[c].astype('category')

for c in ['PATIENT','ENCOUNTER']:
    obs[c] = obs[c].astype('category')   # UUID que repiten -> category (ver nota)

is_num = obs['TYPE'].eq('numeric')
obs['VALUE_num'] = pd.to_numeric(obs['VALUE'].where(is_num), errors='coerce').astype('float32')
obs['VALUE_cat'] = obs['VALUE'].where(~is_num).astype('category')
obs = obs.drop(columns='VALUE')

despues = col_mb(obs)
reduccion = 1 - mb(obs)/base_mb
print(f'Reducción total: {reduccion:.1%}')   # objetivo >=70%

Reducción total: 94.4%


Reducción total: **94.4%** (objetivo ≥ 70%, cumplido).

### Tabla por columna (entregable de la Actividad 2)
Las cinco primeras columnas se arman solas desde `antes`/`despues` y los dtypes. La última —*qué se pierde*— es tuya: completá el dict `notas`.

In [9]:
notas = {   # <- REDACTÁS VOS: qué se pierde en cada conversión
    'DATE':'', 'PATIENT':'', 'ENCOUNTER':'', 'CATEGORY':'', 'CODE':'',
    'DESCRIPTION':'', 'UNITS':'', 'TYPE':'', 'VALUE_num':'', 'VALUE_cat':'',
}

filas, visto_value = [], False
for col in obs.columns:
    en_orig = col in obs_naive.columns
    origen  = col if en_orig else 'VALUE'
    if en_orig:
        mb_antes, dt_antes = antes[col], str(obs_naive[col].dtype)
    elif not visto_value:                       # VALUE_num hereda el peso de VALUE
        mb_antes, dt_antes, visto_value = antes['VALUE'], 'object', True
    else:                                        # VALUE_cat: no re-contar
        mb_antes, dt_antes = 0.0, '(idem VALUE)'
    filas.append({'columna':col, 'dtype_antes':dt_antes, 'dtype_despues':str(obs[col].dtype),
                  'MB_antes':round(float(mb_antes),3), 'MB_despues':round(float(despues[col]),3),
                  'que_se_pierde':notas.get(col,'')})

tabla2 = pd.DataFrame(filas)
tabla2

,columna,dtype_antes,dtype_despues,MB_antes,MB_despues,que_se_pierde
0,DATE,object,"datetime64[ns, UTC]",1197.535,138.845,
1,PATIENT,object,category,1475.224,37.196,
2,ENCOUNTER,object,category,1442.155,148.582,
3,CATEGORY,object,category,989.334,17.356,
4,CODE,object,category,962.296,34.736,
5,DESCRIPTION,object,category,1594.169,34.749,
6,UNITS,object,category,836.831,17.360,
7,TYPE,object,category,951.934,17.356,
8,VALUE_num,object,float32,1004.791,69.422,
9,VALUE_cat,(idem VALUE),category,0.000,72.624,


**Guía para `notas`:** `float32` pierde precisión decimal; `category` estorba operaciones de texto vectorizadas y complica los `merge` (Act. 4); partir `VALUE` obliga a llevar dos columnas con `NaN` complementarios.

**Nota `PATIENT`/`ENCOUNTER`:** son UUID que **repiten** (a diferencia de `patients.Id`, único), por eso `category` los aplasta (~20× frente a dejarlos como texto). El costo es ergonómico en los `merge`: se retoma en la Actividad 4.

## Actividad 3 — Auditoría de calidad

### Faltantes, duplicados y coherencia temporal

In [10]:
# (a) % faltantes por columna, en los tres
for name, df in [('patients',p8),('encounters',enc_naive),('observations',obs)]:
    print(f'== {name} ==\n{(df.isna().mean()*100).round(1)}\n')

# (b) Id de paciente duplicados
print('patients Id duplicados:', p8['Id'].duplicated().sum())

# (c) coherencia temporal
enc = enc_naive.copy()
enc['START'] = pd.to_datetime(enc['START'], utc=True)
ref = p8[['Id','BIRTHDATE','DEATHDATE']].rename(columns={'Id':'PATIENT'})
chk = enc.merge(ref, on='PATIENT', how='left')   # lookup; el merge RIGUROSO es la Act.4
chk['BIRTHDATE'] = pd.to_datetime(chk['BIRTHDATE'], utc=True)
chk['DEATHDATE'] = pd.to_datetime(chk['DEATHDATE'], utc=True)
antes_nacer = (chk['START'] < chk['BIRTHDATE']).sum()
tras_morir  = (chk['START'] > chk['DEATHDATE']).sum()
print('encuentros antes de nacer:', antes_nacer, '| tras defunción:', tras_morir)

== patients ==
Id            0.0
BIRTHDATE     0.0
DEATHDATE    86.9
RACE          0.0
ETHNICITY     0.0
GENDER        0.0
CITY          0.0
STATE         0.0
dtype: float64

== encounters ==
Id                0.0
START             0.0
STOP              0.0
PATIENT           0.0
ENCOUNTERCLASS    0.0
CODE              0.0
DESCRIPTION       0.0
dtype: float64

== observations ==
DATE            0.0
PATIENT         0.0
ENCOUNTER       3.6
CATEGORY        3.6
CODE            0.0
DESCRIPTION     0.0
UNITS          27.3
TYPE            0.0
VALUE_num      38.4
VALUE_cat      61.6
dtype: float64

patients Id duplicados: 0
encuentros antes de nacer: 0 | tras defunción: 3289


### Encuentros posteriores a la defunción
El chequeo marca **3 289** encuentros con `START > DEATHDATE`. Antes de reportarlo como anomalía, un matiz: `DEATHDATE` no tiene hora (queda a medianoche UTC), mientras que `START` es un timestamp completo. Un encuentro el mismo día de la muerte, a cualquier hora > 00:00, cae como 'posterior'. Estratificamos para separar el artefacto de la anomalía real:

In [11]:
mismo_dia = ((chk['START'].dt.normalize() == chk['DEATHDATE'].dt.normalize()) &
             (chk['START'] > chk['DEATHDATE'])).sum()
dias_desp = (chk['START'] > chk['DEATHDATE'] + pd.Timedelta(days=1)).sum()
print('mismo día (artefacto fecha vs timestamp):', mismo_dia)
print('más de 1 día después  :', dias_desp)

mismo día (artefacto fecha vs timestamp): 526
más de 1 día después  : 2763


La estratificación dio 526 el mismo día (artefacto fecha-vs-timestamp) y 2 763 a más de un día — o sea, la mayoría no es artefacto. Antes de concluir hay que ver qué son: cuántos días después, de qué clase, con qué descripción.

In [12]:
post = chk[chk['START'] > chk['DEATHDATE'] + pd.Timedelta(days=1)].copy()
post['dias_tras_muerte'] = (post['START'] - post['DEATHDATE']).dt.days
print(post['dias_tras_muerte'].describe())
print('\nPor clase de encuentro:')
print(post['ENCOUNTERCLASS'].value_counts())
print('\nDescripciones más frecuentes:')
print(post['DESCRIPTION'].value_counts().head(10))

count    2763.000000
mean        6.350706
std         3.764192
min         1.000000
25%         3.000000
50%         6.000000
75%         9.000000
max        14.000000
Name: dias_tras_muerte, dtype: float64

Por clase de encuentro:
ENCOUNTERCLASS
wellness    2763
Name: count, dtype: int64

Descripciones más frecuentes:
DESCRIPTION
Death Certification    2763
Name: count, dtype: int64


De los 3289 encuentros con START posterior a DEATHDATE: 526 son artefacto de
granularidad temporal (mismo día; la defunción se registra a medianoche y el
encuentro tiene hora). Los 2763 restantes son, sin excepción, encuentros
`wellness` de tipo "Death Certification", entre 1 y 14 días tras la muerte
(mediana 6). No son inconsistencias: la certificación ocurre necesariamente
después del fallecimiento.

### **Celda de checkpoint**

In [ ]:
CKPT = BASE / 'ckpt'; CKPT.mkdir(exist_ok=True)
obs.to_parquet(CKPT/'obs.parquet', index=False)
p8.to_parquet(CKPT/'p8.parquet', index=False)
enc_tip.to_parquet(CKPT/'enc_tip.parquet', index=False)
print('checkpoint guardado')

## Actividad 4 — Unir las tres tablas validando cardinalidades

Unimos `observations` → `encounters` → `patients`. El punto no es lograr el join, sino **declarar la cardinalidad esperada antes** y que pandas la verifique.

Dos parámetros nuevos:
- `validate='m:1'` — verifica que la clave del lado **derecho** sea única (many-to-one). Si no lo es, pandas levanta `MergeError` en vez de multiplicar filas en silencio.
- `indicator=True` — agrega una columna `_merge` con `left_only` / `right_only` / `both`, para auditar qué se unió y qué no.

### Cardinalidad esperada (antes de tocar código)

**Join 1 — `observations` → `encounters`** (`obs.ENCOUNTER` = `enc.Id`)
- Cada observación pertenece a **un** encuentro; cada encuentro tiene **muchas** observaciones → **many-to-one** (`validate='m:1'`).
- Filas esperadas: **sin cambio** (≈ 17.36 M), porque con `how='left'` cada observación trae ≤ 1 encuentro.
- Salvedad de la auditoría: **3.6% de `observations` tiene `ENCOUNTER` nulo** → esas filas quedan `left_only`. Es esperado, no un error.

**Join 2 — resultado → `patients`** (`PATIENT` = `patients.Id`)
- Cada fila pertenece a **un** paciente; cada paciente tiene **muchas** filas → **many-to-one** (`validate='m:1'`).
- Filas esperadas: **sin cambio**. `patients.Id` es único (0 duplicados, auditado) → no hay fan-out.
- Esperamos **100% `both`**: toda observación pertenece a un paciente generado.

### Preparación de claves — el costo ergonómico de `category`
 `PATIENT` y `ENCOUNTER` están en `category` (óptimo en memoria), pero un merge sobre `category` exige categorías idénticas en ambos lados o pandas upcastea con warning. Para el join llevamos las claves a un tipo común: `string[pyarrow]` — Arrow-backed, merge limpio y más liviano que `object`. (Si diera problemas de versión, `.astype(str)` siempre funciona.)

> **RAM:** las claves son UUID de alta cardinalidad; materializarlas para el join pesa (~700 MB por columna sobre 17 M filas). Es inherente a unir por UUID: `category` ahorró memoria *en reposo*, pero el join tiene que comparar los valores reales. Llevamos solo las columnas necesarias de cada tabla para no inflar el resultado.

In [13]:
KEY = 'string[pyarrow]'   # tipo común para las claves de join

obs_m = obs.copy()
obs_m['ENCOUNTER'] = obs_m['ENCOUNTER'].astype(KEY)
obs_m['PATIENT']   = obs_m['PATIENT'].astype(KEY)

# encounters: solo lo necesario; Id es la clave derecha del join 1
enc_m = enc_tip[['Id','PATIENT','START','STOP','ENCOUNTERCLASS']].copy()
enc_m['Id']      = enc_m['Id'].astype(KEY)
enc_m['PATIENT'] = enc_m['PATIENT'].astype(KEY)

# patients: renombrar Id -> PATIENT para la clave del join 2
pat_m = p8.rename(columns={'Id':'PATIENT'}).copy()
pat_m['PATIENT'] = pat_m['PATIENT'].astype(KEY)

print('enc.Id duplicados     :', enc_m['Id'].duplicated().sum())        # 0 => ok para m:1
print('pat.PATIENT duplicados:', pat_m['PATIENT'].duplicated().sum())   # 0 => ok para m:1
print('filas observations    :', len(obs_m))

enc.Id duplicados     : 0
pat.PATIENT duplicados: 0
filas observations    : 17355578


### Join 1 — observations → encounters

In [14]:
m1 = obs_m.merge(
    enc_m, left_on='ENCOUNTER', right_on='Id', how='left',
    validate='m:1', indicator=True, suffixes=('', '_enc')
)
print('filas m1 (después):', len(m1))
print(m1['_merge'].value_counts())
m1 = m1.rename(columns={'_merge': '_merge_enc'})   # liberar el nombre para el 2do join

filas m1 (después): 17355578
_merge
both          16731626
left_only       623952
right_only           0
Name: count, dtype: int64


El join respetó la cardinalidad m:1: 17.36 M filas antes y después, sin
multiplicación. Las 623 952 `left_only` (3.6%) son las observaciones con
ENCOUNTER nulo ya detectadas en la auditoría — no matchean porque no tienen
clave, no porque falte el encuentro.




### Chequeo de integridad referencial
Tras el join, hay dos columnas de paciente: `PATIENT` (de observations) y `PATIENT_enc` (del encuentro). Donde hubo match deben coincidir — si no, una observación estaría atribuida a un encuentro de **otro** paciente.

In [15]:
both = m1['_merge_enc'].eq('both')
mismatch = (m1.loc[both, 'PATIENT'] != m1.loc[both, 'PATIENT_enc']).sum()
print('PATIENT inconsistente (obs vs enc):', mismatch)   # esperado 0

m1 = m1.drop(columns=['Id', 'PATIENT_enc'])   # sobran: clave duplicada y PATIENT del encuentro

PATIENT inconsistente (obs vs enc): 0


### Join 2 — (observations+encounters) → patients

In [16]:
m2 = m1.merge(
    pat_m, on='PATIENT', how='left',
    validate='m:1', indicator=True
)
print('filas m2 (después):', len(m2))
print(m2['_merge'].value_counts())

filas m2 (después): 17355578
_merge
both          17355578
left_only            0
right_only           0
Name: count, dtype: int64


Cualquier left_only sería una observación sin paciente en la tabla, pero aquí se observan 100% both. A Synthea no se le ha escapado ninguna.

### Reconciliación de conteos (entregable)


In [17]:
print('observations (inicio):', len(obs))
print('tras join encounters :', len(m1))
print('tras join patients   :', len(m2))
print('\n% match encuentros:', f"{m1['_merge_enc'].eq('both').mean():.1%}")
print('% match pacientes  :', f"{m2['_merge'].eq('both').mean():.1%}")

observations (inicio): 17355578
tras join encounters : 17355578
tras join patients   : 17355578

% match encuentros: 96.4%
% match pacientes  : 100.0%


Las filas se mantuvieron en 17 M en los dos joins (sin multiplicación), lo que confirma las cardinalidades `m:1` declaradas. El único 'faltante' es el `left_only` de encuentros, ya explicado en la auditoría (`ENCOUNTER` nulo). Si `validate` hubiera levantado `MergeError`, ahí estaría *el bug más caro del curso*: una clave que se creía única y no lo era, multiplicando filas en silencio.

## Actividad 5 — Preguntas clínicas

Cuatro preguntas sobre el dataset. **No usamos `m2`**: cada una se responde sobre la tabla mínima (`p8`, `enc_tip`, `obs`), lo que evita el costo de RAM de la tabla unida y deja el código más claro.

> Gotcha de `category`: todos los `groupby` sobre columnas categóricas van con **`observed=True`**. Sin eso, pandas genera un grupo por **cada** categoría existente (p. ej. los 23 018 pacientes) aunque el subconjunto filtrado tenga unos pocos — desperdicia memoria y ensucia el resultado.

### 1. Pacientes por grupo étnico y sexo

Decisión de agrupamiento: en el estándar de EE. UU. (OMB / censo), **raza y etnia son ejes ortogonales** — "hispano" es una etnia, no una raza. Por eso reportamos `RACE × GENDER` y `ETHNICITY × GENDER` por separado, en vez de mezclarlos en una sola variable. Es también como OMOP modela la demografía (`race_concept_id` y `ethnicity_concept_id` separados).

In [18]:
# Raza x sexo
raza_sexo = (p8.groupby(['RACE','GENDER'], observed=True)
               .size().unstack(fill_value=0))
print(raza_sexo, '\n')

# Etnia (hispano / no hispano) x sexo — eje ortogonal a la raza
etnia_sexo = (p8.groupby(['ETHNICITY','GENDER'], observed=True)
                .size().unstack(fill_value=0))
print(etnia_sexo)

GENDER       F     M
RACE                
asian      708   802
black      921   994
hawaiian   162   124
native      63    62
other      136   124
white     9561  9361 

GENDER           F      M
ETHNICITY                
hispanic      1303   1272
nonhispanic  10248  10195


### 2. Encuentros por paciente: media y mediana
Contamos encuentros por paciente y resumimos. Denominador: pacientes **con al menos un encuentro** (los que aparecen en `encounters`).

In [19]:
enc_por_pac = enc_tip.groupby('PATIENT', observed=True).size()

print('media  :', round(enc_por_pac.mean(), 2))
print('mediana:', enc_por_pac.median())
print()
print(enc_por_pac.describe())

media  : 58.9
mediana: 36.0

count    23018.000000
mean        58.900643
std         90.961079
min          1.000000
25%         24.000000
50%         36.000000
75%         57.000000
max        916.000000
dtype: float64


La media suele quedar **por encima** de la mediana: la distribución de encuentros es asimétrica a la derecha (pocos pacientes crónicos con muchísimos encuentros estiran la cola). Para "el paciente típico", la **mediana** representa mejor; la media es sensible a esos outliers.

### 3. Los 10 códigos de observación más frecuentes
`value_counts` sobre `CODE`, y le pegamos la descripción legible (una por código).

In [20]:
top10 = obs['CODE'].value_counts().head(10)

# descripción por código: primera aparición de cada CODE (pareado casi 1:1)
desc = obs.drop_duplicates('CODE').set_index('CODE')['DESCRIPTION']

tbl = top10.rename('frecuencia').to_frame()
tbl['descripcion'] = desc.reindex(tbl.index).astype(str).values
tbl

,frecuencia,descripcion
CODE,,
72514-3,566572,Pain severity - 0-10 verbal numeric rating [Sc...
8480-6,329381,Systolic Blood Pressure
8462-4,329381,Diastolic Blood Pressure
29463-7,313708,Body Weight
8867-4,306907,Heart rate
9279-1,306907,Respiratory rate
8302-2,300653,Body Height
72166-2,299901,Tobacco smoking status
39156-5,278560,Body mass index (BMI) [Ratio]


### 4. Analito elegido: BMI (Body mass index)
Distinto de los ejemplos del README y lo utilizo en mi proyecto de doctorado. Primero ubicamos el **label exacto** buscando entre las categorías de `DESCRIPTION` (no sobre las 17 M filas: buscar en `.cat.categories` es ~300 strings, no millones).

In [21]:
labels = [c for c in obs['DESCRIPTION'].cat.categories if 'body mass index' in c.lower()]
print('labels encontrados:', labels)

mask_bmi = obs['DESCRIPTION'].isin(labels)
bmi = obs[mask_bmi]
print('mediciones de BMI:', len(bmi))

labels encontrados: ['Body mass index (BMI) [Percentile] Per age and sex', 'Body mass index (BMI) [Ratio]']
mediciones de BMI: 335156


#### Distribución y pacientes con ≥ 3 mediciones

In [22]:
vals = bmi['VALUE_num'].dropna()
print(vals.describe(), '\n')

n_por_pac = bmi.groupby('PATIENT', observed=True)['VALUE_num'].count()
print('pacientes con >=3 mediciones de BMI:', (n_por_pac >= 3).sum())

count    335156.000000
mean         32.521183
std          17.746645
min           0.000000
25%          27.200001
50%          28.100000
75%          30.200001
max         100.000000
Name: VALUE_num, dtype: float64 

pacientes con >=3 mediciones de BMI: 22288


#### La media de medias (punto de la actividad)
Dos formas de "promediar" el BMI que **no** dan lo mismo cuando cada paciente tiene distinto número de mediciones:

In [23]:
media_global   = vals.mean()                                   # todas las mediciones pesan igual
media_por_pac  = bmi.groupby('PATIENT', observed=True)['VALUE_num'].mean()
media_de_medias = media_por_pac.mean()                          # cada paciente pesa igual

print('media global (todas las mediciones):', round(media_global, 2))
print('media de medias por paciente       :', round(media_de_medias, 2))

media global (todas las mediciones): 32.52
media de medias por paciente       : 31.9
